# MILP–LCA model reproduction: base case and S0–S5 analyses

This notebook reproduces the base-case optimization and the S0–S5 scenario and sensitivity analyses reported in the manuscript: "Optimizing facility siting, technology selection, and feedstock allocation for plastics recycling using life cycle assessment integrated with mixed-integer linear programming" by:

**Abdullah M.Y. Yaish †, Qian Zhang ‡, Laurence Yang†, P James McLellan,†**

† Department of Chemical Engineering, Queen’s University, Kingston, ON, K7L 3N6, Canada

‡ Robert M. Buchan Department of Mining, Queen’s University, Kingston, ON, K7L 3N6, Canada


The notebook reads the `Optimization formulation` worksheet from the supplementary Excel workbook. The notebook and workbook are expected to be in the same directory.

Requirements: Python, NumPy, pandas, openpyxl, `gurobipy`, and a working Gurobi license.


## Data loading

In [17]:
from pathlib import Path

import numpy as np
import pandas as pd
import gurobipy as gp
from gurobipy import GRB
from IPython.display import display

DATA_FILE = Path("Optimization formulation.xlsx")
SHEET_NAME = "Optimization formulation"

if not DATA_FILE.exists():
    raise FileNotFoundError(
        f"{DATA_FILE.name} was not found. Place the Excel workbook in the same directory as this notebook."
    )

N_PRODUCT = 10
N_TECH = 19
N_PARK = 7
N_UPSTREAM = 90
N_REGION = 34
N_POLYMER = 7
N_IMPACT = 22

N_FACILITY = N_TECH * N_PARK
N_SUPPLY = N_REGION * N_POLYMER
N_TRANSPORT = N_REGION * N_TECH * N_PARK
N_REFERENCE = N_PRODUCT
N_LANDFILL = N_POLYMER

N_VAR = N_FACILITY + N_UPSTREAM + N_SUPPLY + N_TRANSPORT + N_REFERENCE + N_LANDFILL
N_ROW = N_PRODUCT + N_UPSTREAM + N_SUPPLY + N_FACILITY + N_LANDFILL

PARKS = [
    "Sarnia",
    "Nanticoke",
    "Hamilton",
    "Courtice",
    "Cornwall",
    "Sudbury",
    "Sault Ste. Marie",
]
POLYMERS = ["HDPE", "LDPE", "LLDPE", "PET", "PP", "PS", "PVC"]

MECHANICAL_RECS = [1, 2, 3, 4]
NONMECHANICAL_RECS = list(range(5, 20))
DISSOLUTION_RECS = [5, 6, 7, 8, 9, 10]
ADVANCED_PET_RECS = [11, 12, 13, 14]
PYROLYSIS_RECS = [15, 16, 17, 18]
INCINERATION_RECS = [19]

POLYMER_RECS = {
    "HDPE": [1, 5],
    "LDPE": [2, 6],
    "LLDPE": [],
    "PET": [3, 7, 11, 12, 13, 14],
    "PP": [4, 8],
    "PS": [9],
    "PVC": [10],
}

J_FAC = slice(0, N_FACILITY)
J_UP = slice(J_FAC.stop, J_FAC.stop + N_UPSTREAM)
J_SUP = slice(J_UP.stop, J_UP.stop + N_SUPPLY)
J_TR = slice(J_SUP.stop, J_SUP.stop + N_TRANSPORT)
J_REF = slice(J_TR.stop, J_TR.stop + N_REFERENCE)
J_LF = slice(J_REF.stop, J_REF.stop + N_LANDFILL)

I_PRODUCT = slice(0, N_PRODUCT)
I_UP = slice(I_PRODUCT.stop, I_PRODUCT.stop + N_UPSTREAM)
I_SUP = slice(I_UP.stop, I_UP.stop + N_SUPPLY)
I_TR = slice(I_SUP.stop, I_SUP.stop + N_FACILITY)
I_CL = slice(I_TR.stop, I_TR.stop + N_LANDFILL)


def _numeric_block(df):
    return df.apply(pd.to_numeric, errors="coerce").fillna(0.0).to_numpy(dtype=float)


def _bound_vector(series, default):
    values = []
    for value in series:
        if pd.isna(value) or str(value).strip() == "":
            values.append(default)
            continue

        text = str(value).strip().lower()
        if text in {"+inf", "inf", "+infinity", "infinity"}:
            values.append(np.inf)
        elif text in {"-inf", "-infinity"}:
            values.append(-np.inf)
        else:
            values.append(float(value))
    return np.asarray(values, dtype=float)


raw = pd.read_excel(
    DATA_FILE,
    sheet_name=SHEET_NAME,
    header=None,
    usecols="A:GJL",
    nrows=516,
    engine="openpyxl",
)

if raw.shape[0] < 516 or raw.shape[1] < 5004:
    raise ValueError("The Optimization formulation worksheet does not match the expected published layout.")

COL_LABELS = raw.iloc[1, 1:5001].tolist()
ROW_LABELS = raw.iloc[2:480, 0].tolist()

A = _numeric_block(raw.iloc[2:480, 1:5001])
F = pd.to_numeric(raw.iloc[2:480, 5003], errors="raise").to_numpy(dtype=float)
ROW_SENSES = raw.iloc[2:480, 5002].astype(str).str.strip().tolist()

LB = _bound_vector(raw.iloc[488, 1:5001], default=0.0)
UB = _bound_vector(raw.iloc[489, 1:5001], default=np.inf)

H = _numeric_block(raw.iloc[494:516, 1:5001])
NORMALIZATION = pd.to_numeric(raw.iloc[494:516, 5002], errors="raise").to_numpy(dtype=float)
WEIGHTS = pd.to_numeric(raw.iloc[494:516, 5003], errors="raise").to_numpy(dtype=float)

if A.shape != (N_ROW, N_VAR):
    raise ValueError(f"A has shape {A.shape}; expected {(N_ROW, N_VAR)}.")
if H.shape != (N_IMPACT, N_VAR):
    raise ValueError(f"H has shape {H.shape}; expected {(N_IMPACT, N_VAR)}.")
if len(F) != N_ROW or len(LB) != N_VAR or len(UB) != N_VAR:
    raise ValueError("Vector dimensions do not match the published model.")
if not np.isfinite(A).all() or not np.isfinite(H).all() or not np.isfinite(F).all():
    raise ValueError("A, H, and f must contain finite numeric values.")
if not np.isfinite(NORMALIZATION).all() or np.any(np.abs(NORMALIZATION) < 1e-30):
    raise ValueError("Normalization factors are invalid.")
if not np.isfinite(WEIGHTS).all():
    raise ValueError("Weighting factors are invalid.")

expected_landfill_labels = [f"Landfilled {p} [kg]" for p in POLYMERS]
if [str(x).strip() for x in COL_LABELS[J_LF]] != expected_landfill_labels:
    raise ValueError("The landfill-variable block does not match the published workbook.")

BALE_INDEX = {}
for polymer in POLYMERS:
    label = f"{polymer} bale [kg]"
    matches = [j for j, x in enumerate(COL_LABELS) if str(x).strip() == label]
    if len(matches) != 1:
        raise ValueError(f"Expected one column labelled '{label}', found {len(matches)}.")
    BALE_INDEX[polymer] = matches[0]

for p, polymer in enumerate(POLYMERS):
    j0 = J_SUP.start + p * N_REGION
    j1 = j0 + N_REGION
    labels = [str(x) for x in COL_LABELS[j0:j1]]
    if not all(label.startswith(f"{polymer} Sourcing ") for label in labels):
        raise ValueError(f"Regional supply block for {polymer} is not in the expected position.")

print(f"Loaded {DATA_FILE.name}")
print(f"A: {A.shape} | H: {H.shape} | variables: {N_VAR} | constraints: {N_ROW}")

Loaded Optimization formulation.xlsx
A: (478, 5000) | H: (22, 5000) | variables: 5000 | constraints: 478


## Model construction and solution

In [19]:
def facility_index(park, rec):
    return (park - 1) * N_TECH + (rec - 1)


def objective_coefficients(H_matrix):
    return (WEIGHTS / NORMALIZATION) @ H_matrix


def _apply_supply_scaling(f_eff, ub_eff, factor):
    for p, polymer in enumerate(POLYMERS):
        j0 = J_SUP.start + p * N_REGION
        j1 = j0 + N_REGION

        segment = ub_eff[j0:j1].copy()
        finite = np.isfinite(segment)
        segment[finite] *= factor
        ub_eff[j0:j1] = segment

        if not np.isfinite(segment).all():
            raise ValueError(f"Regional supply bounds for {polymer} must be finite.")

        total_available = float(segment.sum())
        ub_eff[BALE_INDEX[polymer]] = total_available
        f_eff[I_CL.start + p] = total_available


def _add_matrix_row(model, variables, row, sense, rhs, name):
    nz = np.flatnonzero(row)
    expr = gp.quicksum(float(row[j]) * variables[int(j)] for j in nz)
    sense = str(sense).strip()

    if sense in {"≥", ">", ">=", "G", "g"}:
        model.addConstr(expr >= float(rhs), name=name)
    elif sense in {"≤", "<", "<=", "L", "l"}:
        model.addConstr(expr <= float(rhs), name=name)
    elif sense in {"=", "==", "E", "e", "EQ", "eq"}:
        model.addConstr(expr == float(rhs), name=name)
    else:
        raise ValueError(f"Unknown row sense '{sense}' for {name}.")


def solve_case(case, hotspot_columns=None):
    f_eff = F.copy()
    lb_eff = LB.copy()
    ub_eff = UB.copy()
    H_eff = H.copy()

    supply_factor = float(case.get("supply_factor", 1.0))
    _apply_supply_scaling(f_eff, ub_eff, supply_factor)

    h_factor = case.get("h_factor")
    if h_factor is not None:
        if hotspot_columns is None:
            raise ValueError("S4 requires the base-case hotspot columns.")
        H_eff[:, hotspot_columns] *= float(h_factor)

    park_cap = int(case.get("park_cap", 2))
    polymer_park_cap = int(case.get("polymer_park_cap", 2))

    model = gp.Model(f"MPW_MILP_LCA_{case['id']}")
    model.Params.OutputFlag = 0
    model.Params.MIPGap = 0.0

    variables = []
    for j in range(N_VAR):
        vtype = GRB.BINARY if j < N_FACILITY else GRB.CONTINUOUS
        lb_j = -GRB.INFINITY if np.isneginf(lb_eff[j]) else float(lb_eff[j])
        ub_j = GRB.INFINITY if np.isposinf(ub_eff[j]) else float(ub_eff[j])
        variables.append(model.addVar(lb=lb_j, ub=ub_j, vtype=vtype, name=f"s[{j}]"))

    model.update()

    for i in range(N_ROW):
        _add_matrix_row(model, variables, A[i], ROW_SENSES[i], f_eff[i], f"row_{i}")

    for park in range(1, N_PARK + 1):
        model.addConstr(
            gp.quicksum(variables[facility_index(park, rec)] for rec in range(1, N_TECH + 1))
            <= park_cap,
            name=f"park_cap_{park}",
        )

    for polymer, recs in POLYMER_RECS.items():
        if not recs:
            continue
        for park in range(1, N_PARK + 1):
            model.addConstr(
                gp.quicksum(variables[facility_index(park, rec)] for rec in recs)
                <= polymer_park_cap,
                name=f"{polymer}_park_cap_{park}",
            )

    nonmechanical_min = case.get("nonmechanical_min")
    if nonmechanical_min is not None:
        model.addConstr(
            gp.quicksum(
                variables[facility_index(park, rec)]
                for park in range(1, N_PARK + 1)
                for rec in NONMECHANICAL_RECS
            )
            >= int(nonmechanical_min),
            name="nonmechanical_min",
        )

    mechanical_max = case.get("mechanical_max")
    if mechanical_max is not None:
        model.addConstr(
            gp.quicksum(
                variables[facility_index(park, rec)]
                for park in range(1, N_PARK + 1)
                for rec in MECHANICAL_RECS
            )
            <= int(mechanical_max),
            name="mechanical_max",
        )

    family_recs = case.get("family_recs")
    family_min = case.get("family_min")
    if family_recs is not None and family_min is not None:
        model.addConstr(
            gp.quicksum(
                variables[facility_index(park, rec)]
                for park in range(1, N_PARK + 1)
                for rec in family_recs
            )
            >= int(family_min),
            name="family_min",
        )

    obj_coeff = objective_coefficients(H_eff)
    if not np.isfinite(obj_coeff).all():
        raise ValueError("Objective coefficients contain non-finite values.")

    nz_obj = np.flatnonzero(obj_coeff)
    model.setObjective(
        gp.quicksum(float(obj_coeff[j]) * variables[int(j)] for j in nz_obj),
        GRB.MINIMIZE,
    )

    model.optimize()

    if model.Status != GRB.OPTIMAL:
        raise RuntimeError(f"{case['id']} did not solve to optimality. Gurobi status: {model.Status}")

    solution = np.array([v.X for v in variables], dtype=float)
    selected = np.flatnonzero(solution[J_FAC] > 0.5)

    selected_facilities = [
        {
            "Park": PARKS[j // N_TECH],
            "REC": (j % N_TECH) + 1,
        }
        for j in selected
    ]

    rec_counts = {}
    for j in selected:
        rec = (j % N_TECH) + 1
        rec_counts[rec] = rec_counts.get(rec, 0) + 1

    processed_kg = float(solution[J_SUP].sum())
    landfill_kg = float(solution[J_LF].sum())
    available_kg = float(f_eff[I_CL].sum())

    return {
        "id": case["id"],
        "family": case["family"],
        "description": case["description"],
        "objective": float(model.ObjVal),
        "solution": solution,
        "objective_coefficients": obj_coeff,
        "selected_facilities": selected_facilities,
        "rec_counts": rec_counts,
        "facility_count": len(selected_facilities),
        "processed_kg": processed_kg,
        "landfill_kg": landfill_kg,
        "available_kg": available_kg,
        "landfill_percent": 100.0 * landfill_kg / available_kg,
        "objective_per_tonne_available": float(model.ObjVal) / (available_kg / 1000.0),
        "objective_per_tonne_processed": float(model.ObjVal) / (processed_kg / 1000.0),
        "h_factor": h_factor,
        "supply_factor": supply_factor,
    }


def select_s4_hotspots(base_result):
    contributions = base_result["solution"] * base_result["objective_coefficients"]
    absolute = np.abs(contributions)
    order = np.argsort(-absolute)

    total = absolute.sum()
    if total <= 0:
        raise ValueError("Base-case absolute objective contribution is zero.")

    cumulative = np.cumsum(absolute[order]) / total
    n_for_80_percent = int(np.searchsorted(cumulative, 0.80, side="left") + 1)
    n_select = min(max(20, n_for_80_percent), 30)
    selected = order[:n_select]

    if len(selected) != 20:
        raise RuntimeError(
            f"The published S4 setup targets 20 columns, but the current workbook selected {len(selected)}."
        )

    table = pd.DataFrame(
        {
            "Rank": np.arange(1, len(selected) + 1),
            "Column index (0-based)": selected,
            "Variable": [COL_LABELS[j] for j in selected],
            "Base contribution": contributions[selected],
            "Absolute-driver share (%)": 100.0 * absolute[selected] / total,
        }
    )
    return selected, table

## Base case (S0)

In [21]:
BASE_CASE = {
    "id": "S0",
    "family": "S0",
    "description": "Base case",
}

base_result = solve_case(BASE_CASE)
S4_HOTSPOT_COLUMNS, S4_HOTSPOT_TABLE = select_s4_hotspots(base_result)

print(f"Objective: {base_result['objective'] / 1e9:.12f} × 10^9")
print(f"Selected facilities: {base_result['facility_count']}")
print(f"Processed feedstock: {base_result['processed_kg'] / 1e6:.3f} kt/y")
print(f"Residual landfill: {base_result['landfill_kg'] / 1e6:.3f} kt/y")
print(f"S4 hotspot columns: {len(S4_HOTSPOT_COLUMNS)}")

display(pd.DataFrame(base_result["selected_facilities"]))

Set parameter Username
Academic license - for non-commercial use only - expires 2027-07-17
Objective: 16.481826660259 × 10^9
Selected facilities: 14
Processed feedstock: 700.000 kt/y
Residual landfill: 278.465 kt/y
S4 hotspot columns: 20


,Park,REC
0,Sarnia,3
1,Sarnia,17
2,Nanticoke,1
3,Nanticoke,4
4,Hamilton,2
5,Hamilton,9
6,Courtice,1
7,Courtice,2
8,Cornwall,3
9,Cornwall,17


## S1–S5 scenario and sensitivity analyses

In [23]:
SCENARIO_CASES = [
    {"id": "S1-L", "family": "S1", "description": "Portfolio transition constraint (low)",
     "nonmechanical_min": 6, "mechanical_max": 8},
    {"id": "S1-M", "family": "S1", "description": "Portfolio transition constraint (moderate)",
     "nonmechanical_min": 7, "mechanical_max": 7},
    {"id": "S1-S", "family": "S1", "description": "Portfolio transition constraint (strong)",
     "nonmechanical_min": 8, "mechanical_max": 6},

    {"id": "S2A-L", "family": "S2A", "description": "Dissolution emphasis (low)",
     "family_recs": DISSOLUTION_RECS, "family_min": 3},
    {"id": "S2A-M", "family": "S2A", "description": "Dissolution emphasis (moderate)",
     "family_recs": DISSOLUTION_RECS, "family_min": 5},
    {"id": "S2A-S", "family": "S2A", "description": "Dissolution emphasis (strong)",
     "family_recs": DISSOLUTION_RECS, "family_min": 7},

    {"id": "S2B-L", "family": "S2B", "description": "Advanced PET emphasis (low)",
     "family_recs": ADVANCED_PET_RECS, "family_min": 1},
    {"id": "S2B-M", "family": "S2B", "description": "Advanced PET emphasis (moderate)",
     "family_recs": ADVANCED_PET_RECS, "family_min": 2},
    {"id": "S2B-S", "family": "S2B", "description": "Advanced PET emphasis (strong)",
     "family_recs": ADVANCED_PET_RECS, "family_min": 3},

    {"id": "S2C-L", "family": "S2C", "description": "MPW catalytic fast pyrolysis emphasis (low)",
     "family_recs": PYROLYSIS_RECS, "family_min": 3},
    {"id": "S2C-M", "family": "S2C", "description": "MPW catalytic fast pyrolysis emphasis (moderate)",
     "family_recs": PYROLYSIS_RECS, "family_min": 4},
    {"id": "S2C-S", "family": "S2C", "description": "MPW catalytic fast pyrolysis emphasis (strong)",
     "family_recs": PYROLYSIS_RECS, "family_min": 5},

    {"id": "S2D-L", "family": "S2D", "description": "Incineration placement (low)",
     "family_recs": INCINERATION_RECS, "family_min": 1},
    {"id": "S2D-M", "family": "S2D", "description": "Incineration placement (moderate)",
     "family_recs": INCINERATION_RECS, "family_min": 2},
    {"id": "S2D-S", "family": "S2D", "description": "Incineration placement (strong)",
     "family_recs": INCINERATION_RECS, "family_min": 3},

    {"id": "S3-L", "family": "S3", "description": "Combined siting flexibility (low)",
     "park_cap": 3, "polymer_park_cap": 3},
    {"id": "S3-M", "family": "S3", "description": "Combined siting flexibility (moderate)",
     "park_cap": 5, "polymer_park_cap": 4},
    {"id": "S3-S", "family": "S3", "description": "Combined siting flexibility (strong)",
     "park_cap": 8, "polymer_park_cap": 5},

    {"id": "S4-L", "family": "S4", "description": "Targeted H hotspot sensitivity low",
     "h_factor": 0.8},
    {"id": "S4-B", "family": "S4", "description": "Targeted H hotspot sensitivity baseline",
     "h_factor": 1.0},
    {"id": "S4-H", "family": "S4", "description": "Targeted H hotspot sensitivity high",
     "h_factor": 1.2},

    {"id": "S5-L", "family": "S5", "description": "Supply availability scaling (low)",
     "supply_factor": 0.8},
    {"id": "S5-B", "family": "S5", "description": "Supply availability scaling (baseline)",
     "supply_factor": 1.0},
    {"id": "S5-H", "family": "S5", "description": "Supply availability scaling (high)",
     "supply_factor": 1.2},
]

results = [base_result]

for case in SCENARIO_CASES:
    result = solve_case(case, hotspot_columns=S4_HOTSPOT_COLUMNS)
    results.append(result)
    print(f"{case['id']}: solved")

base_objective = base_result["objective"]
base_per_available = base_result["objective_per_tonne_available"]
base_per_processed = base_result["objective_per_tonne_processed"]

summary_rows = []
for result in results:
    rec_counts = ", ".join(
        f"REC {rec}: {count}" for rec, count in sorted(result["rec_counts"].items())
    )

    summary_rows.append(
        {
            "Scenario": result["id"],
            "Family": result["family"],
            "Description": result["description"],
            "Objective (×10^9)": result["objective"] / 1e9,
            "Change vs S0 (%)": 100.0 * (result["objective"] - base_objective) / base_objective,
            "Facilities": result["facility_count"],
            "REC counts": rec_counts,
            "Available feedstock (kt)": result["available_kg"] / 1e6,
            "Processed feedstock (kt)": result["processed_kg"] / 1e6,
            "Residual landfill (kt)": result["landfill_kg"] / 1e6,
            "Residual landfill (%)": result["landfill_percent"],
            "H factor": result["h_factor"] if result["h_factor"] is not None else np.nan,
            "H target columns": len(S4_HOTSPOT_COLUMNS) if result["family"] == "S4" else np.nan,
            "Objective / t available": result["objective_per_tonne_available"],
            "Change vs S0, objective / t available (%)":
                100.0 * (result["objective_per_tonne_available"] - base_per_available) / base_per_available,
            "Objective / t processed": result["objective_per_tonne_processed"],
            "Change vs S0, objective / t processed (%)":
                100.0 * (result["objective_per_tonne_processed"] - base_per_processed) / base_per_processed,
        }
    )

results_table = pd.DataFrame(summary_rows)
display(results_table)

S1-L: solved
S1-M: solved
S1-S: solved
S2A-L: solved
S2A-M: solved
S2A-S: solved
S2B-L: solved
S2B-M: solved
S2B-S: solved
S2C-L: solved
S2C-M: solved
S2C-S: solved
S2D-L: solved
S2D-M: solved
S2D-S: solved
S3-L: solved
S3-M: solved
S3-S: solved
S4-L: solved
S4-B: solved
S4-H: solved
S5-L: solved
S5-B: solved
S5-H: solved


,Scenario,Family,Description,Objective (×10^9),Change vs S0 (%),Facilities,REC counts,Available feedstock (kt),Processed feedstock (kt),Residual landfill (kt),Residual landfill (%),H factor,H target columns,Objective / t available,"Change vs S0, objective / t available (%)",Objective / t processed,"Change vs S0, objective / t processed (%)"
0,S0,S0,Base case,16.481827,0.000000,14,"REC 1: 3, REC 2: 2, REC 3: 3, REC 4: 2, REC 9:...",978.464913,700.000000,278.464913,28.459366,NaN,NaN,16844.576070,0.000000,23545.466658,0.000000
1,S1-L,S1,Portfolio transition constraint (low),17.006998,3.186366,14,"REC 1: 3, REC 2: 2, REC 3: 1, REC 4: 2, REC 9:...",978.464913,700.000000,278.464913,28.459366,NaN,NaN,17381.305940,3.186366,24295.711437,3.186366
2,S1-M,S1,Portfolio transition constraint (moderate),17.269584,4.779549,14,"REC 1: 3, REC 2: 2, REC 4: 2, REC 9: 2, REC 12...",978.464913,700.000000,278.464913,28.459366,NaN,NaN,17649.670875,4.779549,24670.833827,4.779549
3,S1-S,S1,Portfolio transition constraint (strong),17.554150,6.506097,14,"REC 1: 3, REC 2: 1, REC 4: 2, REC 9: 2, REC 12...",978.464913,700.000000,278.464913,28.459366,NaN,NaN,17940.500610,6.506097,25077.357672,6.506097
4,S2A-L,S2A,Dissolution emphasis (low),16.768987,1.742284,14,"REC 1: 3, REC 2: 2, REC 3: 3, REC 4: 2, REC 9:...",978.464913,700.000000,278.464913,28.459366,NaN,NaN,17138.056448,1.742284,23955.695591,1.742284
5,S2A-M,S2A,Dissolution emphasis (moderate),18.028585,9.384632,14,"REC 1: 3, REC 3: 3, REC 4: 2, REC 6: 2, REC 9:...",978.464913,700.000000,278.464913,28.459366,NaN,NaN,18425.377481,9.384632,25755.121966,9.384632
6,S2A-S,S2A,Dissolution emphasis (strong),19.440475,17.950976,14,"REC 1: 3, REC 3: 3, REC 6: 2, REC 8: 2, REC 9:...",978.464913,700.000000,278.464913,28.459366,NaN,NaN,19868.341927,17.950976,27772.107796,17.950976
7,S2B-L,S2B,Advanced PET emphasis (low),16.744412,1.593183,14,"REC 1: 3, REC 2: 2, REC 3: 2, REC 4: 2, REC 9:...",978.464913,700.000000,278.464913,28.459366,NaN,NaN,17112.941005,1.593183,23920.589047,1.593183
8,S2B-M,S2B,Advanced PET emphasis (moderate),17.006998,3.186366,14,"REC 1: 3, REC 2: 2, REC 3: 1, REC 4: 2, REC 9:...",978.464913,700.000000,278.464913,28.459366,NaN,NaN,17381.305940,3.186366,24295.711437,3.186366
9,S2B-S,S2B,Advanced PET emphasis (strong),17.269584,4.779549,14,"REC 1: 3, REC 2: 2, REC 4: 2, REC 9: 2, REC 12...",978.464913,700.000000,278.464913,28.459366,NaN,NaN,17649.670875,4.779549,24670.833827,4.779549
